# SAFOD solid-Earth tide model — June 17, 2026

This notebook is self-contained: it needs only **NumPy** and **Matplotlib**.

It models the first-pass scalar solid-Earth tide at SAFOD from the Sun and Moon, then plots the result over the active-source experiment window.

The upper plot is **tidal strain**, not seismic δv/v. Its vertical axis is **nanostrain**:

```text
1 nanostrain = 10⁻⁹ dimensionless strain
```

The lower panel is a normalized time-shape template for fitting against a measured apparent δv/v time series.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np

OUT_DIR = Path.cwd()

# SAFOD coordinates: latitude north, longitude east-positive.
LATITUDE_DEG = 35.9746
LONGITUDE_DEG = -120.5516

# Actual experiment window from the AWD reconstruction.
START_UTC = datetime(2026, 6, 16, 23, 48, tzinfo=timezone.utc)
END_UTC = datetime(2026, 6, 17, 23, 44, tzinfo=timezone.utc)
SAMPLE_MINUTES = 1

# Constants and Love numbers used by the first-pass model.
G = 6.674e-11
EARTH_RADIUS_M = 6.371e6
SURFACE_GRAVITY_M_S2 = 9.81
MOON_MASS_KG = 7.342e22
SUN_MASS_KG = 1.989e30
H2 = 0.6032
L2 = 0.0839
AREAL_LOVE_FACTOR = 2.0 * H2 - 6.0 * L2
ASTRONOMICAL_UNIT_M = 1.495978707e11

print(f"Love-number factor 2h₂ − 6l₂ = {AREAL_LOVE_FACTOR:.4f}")

## Equations and units

The model calculates the degree-2 tide-generating potential:

```text
W_b(t) = [G M_b / D_b(t)³] Rₑ² P₂(cos θ_b(t))

P₂(x) = (3x² − 1) / 2
```

`W` has units of m²/s². Dividing by `g Rₑ` makes it dimensionless. The scalar strain proxy is:

```text
ε_tide(t) = (2h₂ − 6l₂) W(t) / [g Rₑ]
```

The plot displays `ε_tide × 10⁹`, which is nanostrain. For example, 50 nanostrain means 50 × 10⁻⁹ = 5 × 10⁻⁸ strain.

This is a forcing/template model. The measured apparent velocity response requires an unknown coupling:

```text
(δv/v)_app(t) = S ε_tide(t) + drift(t) + noise(t)
```

The model does not assume `S = 1`.

In [ ]:
def datetime_range(start, end, step_minutes):
    count = int((end - start).total_seconds() // (60.0 * step_minutes))
    return [start + timedelta(minutes=step_minutes * i) for i in range(count + 1)]


def julian_date(times):
    unix_seconds = np.array([t.timestamp() for t in times], dtype=float)
    return unix_seconds / 86400.0 + 2440587.5


def sun_position(jd):
    d = jd - 2451545.0
    mean_longitude = np.deg2rad((280.460 + 0.9856474 * d) % 360.0)
    mean_anomaly = np.deg2rad((357.528 + 0.9856003 * d) % 360.0)
    ecliptic_longitude = (
        mean_longitude
        + np.deg2rad(1.915) * np.sin(mean_anomaly)
        + np.deg2rad(0.020) * np.sin(2.0 * mean_anomaly)
    )
    distance_au = (
        1.00014
        - 0.01671 * np.cos(mean_anomaly)
        - 0.00014 * np.cos(2.0 * mean_anomaly)
    )
    obliquity = np.deg2rad(23.4393 - 3.563e-7 * d)
    ra = np.arctan2(np.cos(obliquity) * np.sin(ecliptic_longitude),
                    np.cos(ecliptic_longitude))
    dec = np.arcsin(np.sin(obliquity) * np.sin(ecliptic_longitude))
    return ra, dec, distance_au * ASTRONOMICAL_UNIT_M


def moon_position(jd):
    # Compact low-precision lunar orbital elements.
    d = jd - 2451543.5
    node = np.deg2rad((125.1228 - 0.0529538083 * d) % 360.0)
    inclination = np.deg2rad(5.1454)
    perigee = np.deg2rad((318.0634 + 0.1643573223 * d) % 360.0)
    semimajor_axis = 60.2666
    eccentricity = 0.054900
    mean_anomaly = np.deg2rad((115.3654 + 13.0649929509 * d) % 360.0)

    eccentric_anomaly = mean_anomaly.copy()
    for _ in range(3):
        eccentric_anomaly -= (
            eccentric_anomaly - eccentricity * np.sin(eccentric_anomaly)
            - mean_anomaly
        ) / (1.0 - eccentricity * np.cos(eccentric_anomaly))

    x_orbit = semimajor_axis * (np.cos(eccentric_anomaly) - eccentricity)
    y_orbit = semimajor_axis * np.sqrt(1.0 - eccentricity**2) * np.sin(eccentric_anomaly)
    radius = np.hypot(x_orbit, y_orbit)
    true_anomaly = np.arctan2(y_orbit, x_orbit)
    argument = true_anomaly + perigee

    x_ecl = radius * (np.cos(node) * np.cos(argument)
                      - np.sin(node) * np.sin(argument) * np.cos(inclination))
    y_ecl = radius * (np.sin(node) * np.cos(argument)
                      + np.cos(node) * np.sin(argument) * np.cos(inclination))
    z_ecl = radius * np.sin(argument) * np.sin(inclination)

    obliquity = np.deg2rad(23.4393 - 3.563e-7 * (jd - 2451545.0))
    x_eq = x_ecl
    y_eq = y_ecl * np.cos(obliquity) - z_ecl * np.sin(obliquity)
    z_eq = y_ecl * np.sin(obliquity) + z_ecl * np.cos(obliquity)
    ra = np.arctan2(y_eq, x_eq)
    dec = np.arctan2(z_eq, np.hypot(x_eq, y_eq))
    return ra, dec, radius * EARTH_RADIUS_M


def cos_zenith_angle(jd, ra, dec):
    d = jd - 2451545.0
    lst = np.deg2rad((280.46061837 + 360.98564736629 * d + LONGITUDE_DEG) % 360.0)
    hour_angle = lst - ra
    latitude = np.deg2rad(LATITUDE_DEG)
    return (np.sin(latitude) * np.sin(dec)
            + np.cos(latitude) * np.cos(dec) * np.cos(hour_angle))


def degree_two_potential(mass_kg, distance_m, cos_zenith):
    p2 = 0.5 * (3.0 * cos_zenith**2 - 1.0)
    return G * mass_kg / distance_m**3 * EARTH_RADIUS_M**2 * p2


def model_tide(times):
    jd = julian_date(times)
    sun_ra, sun_dec, sun_distance = sun_position(jd)
    moon_ra, moon_dec, moon_distance = moon_position(jd)
    sun_potential = degree_two_potential(
        SUN_MASS_KG, sun_distance, cos_zenith_angle(jd, sun_ra, sun_dec)
    )
    moon_potential = degree_two_potential(
        MOON_MASS_KG, moon_distance, cos_zenith_angle(jd, moon_ra, moon_dec)
    )
    scale = AREAL_LOVE_FACTOR / (SURFACE_GRAVITY_M_S2 * EARTH_RADIUS_M)
    return {
        "sun_strain": scale * sun_potential,
        "moon_strain": scale * moon_potential,
        "total_strain": scale * (sun_potential + moon_potential),
    }

In [ ]:
times = datetime_range(START_UTC, END_UTC, SAMPLE_MINUTES)
result = model_tide(times)
total = result["total_strain"]

print(f"Window: {times[0].isoformat()} → {times[-1].isoformat()}")
print(f"Samples: {len(times)}")
print(f"Minimum total strain: {total.min():.6e}")
print(f"Maximum total strain: {total.max():.6e}")
print(f"Peak-to-peak: {np.ptp(total):.6e} = {np.ptp(total) * 1e9:.2f} nanostrain")
print(f"Half-range: {0.5 * np.ptp(total):.6e} = {0.5 * np.ptp(total) * 1e9:.2f} nanostrain")

In [ ]:
time_array = np.array(times)
nanostrain = 1e9
total = result["total_strain"]
mean_removed = total - total.mean()
template = mean_removed / np.max(np.abs(mean_removed))

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
axes[0].plot(time_array, result["sun_strain"] * nanostrain, label="Sun", lw=1.8)
axes[0].plot(time_array, result["moon_strain"] * nanostrain, label="Moon", lw=1.8)
axes[0].plot(time_array, total * nanostrain, label="Sun + Moon", color="black", lw=2.2)
axes[0].axhline(0.0, color="0.5", lw=0.7)
axes[0].set_ylabel("nanostrain (10^-9 strain)\n[dimensionless strain]")
axes[0].set_title(
    "First-pass solid-Earth tide at SAFOD\n"
    "35.9746°N, 120.5516°W — June 17, 2026 experiment window\n"
    "This is tidal strain, not seismic δv/v"
)
axes[0].legend(frameon=False, ncol=3, loc="upper right")
axes[0].grid(alpha=0.2)

axes[1].plot(time_array, template, color="#8c2d04", lw=2.0)
axes[1].axhline(0.0, color="0.5", lw=0.7)
axes[1].set_ylabel("normalized template")
axes[1].set_xlabel("UTC")
axes[1].set_ylim(-1.1, 1.1)
axes[1].grid(alpha=0.2)
axes[1].text(
    0.01, 0.08,
    "Template = (ε_tide − mean) / max|ε_tide − mean|\n"
    "Use this shape when fitting amplitude A in δv/v",
    transform=axes[1].transAxes, fontsize=9, color="#8c2d04",
    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)
axes[1].xaxis.set_major_locator(mdates.HourLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
for label in axes[1].get_xticklabels():
    label.set_rotation(30)
    label.set_ha("right")

plot_path = OUT_DIR / "safod_solid_earth_tide_2026-06-17.png"
fig.savefig(plot_path, dpi=220)
plt.show()
print(f"Saved {plot_path}")

In [ ]:
# Optional: save the modeled series for use in a later δv/v fit.
import csv

csv_path = OUT_DIR / "safod_solid_earth_tide_2026-06-17.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["utc", "total_strain", "sun_strain", "moon_strain"])
    for t, total_i, sun_i, moon_i in zip(
        times, result["total_strain"], result["sun_strain"], result["moon_strain"]
    ):
        writer.writerow([t.isoformat(), total_i, sun_i, moon_i])
print(f"Saved {csv_path}")

## Interpretation

The calculated strain is only about tens of nanostrain. That is the predicted astronomical forcing in this scalar first-pass model.

To compare it with the measured active-source velocity-change series, fit a model such as:

```text
y(t) = A τ(t) + c₀ + c₁t + noise(t)
```

Here `A` has units of fractional velocity change, while `τ(t)` is unitless. The ratio between a fitted `A` and the modeled strain amplitude is an empirical strain sensitivity. It is not determined by the Sun/Moon calculation.

This model also omits the full downhole strain tensor, fiber projection, ocean loading, atmospheric loading, hydrology, and local fault constitutive effects. It is therefore a transparent forcing/template calculation rather than a complete prediction of the DAS response.